# ViennaRNA Secondary Structure Prediction

This notebook demonstrates RNA secondary structure prediction using ViennaRNA. ViennaRNA uses thermodynamic nearest-neighbor models to compute the minimum free energy (MFE) fold for an RNA sequence, returning the predicted structure in dot-bracket notation along with the MFE in kcal/mol. Because it is a CPU-only tool with no model weights to load, predictions complete in milliseconds even for long sequences. The example folds five short RNA sequences that span a range of structural behaviors, from stable hairpins to unstructured stretches.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("viennarna")
display_overview("viennarna")
display_docs_section("viennarna", "Background")

# ViennaRNA

> ViennaRNA is a fast RNA secondary structure prediction tool that uses thermodynamic parameters to compute the [minimum free energy](https://en.wikipedia.org/wiki/Nucleic_acid_thermodynamics) (MFE) structure from nucleotide sequences.

## Background

**What does this tool measure/predict?**
ViennaRNA predicts the secondary structure of RNA molecules; the pattern of [Watson-Crick](https://en.wikipedia.org/wiki/Watson%E2%80%93Crick_base_pair) and wobble base pairs that form within a single RNA strand. It outputs the structure in dot-bracket notation along with the minimum free energy (MFE).

**Why is this important?**
RNA secondary structure is fundamental to understanding RNA function:

* **mRNA stability:** Secondary structures in UTRs affect translation efficiency and mRNA half-life
* **Regulatory elements:** Riboswitches, thermosensors, and other regulatory RNAs depend on specific structures
* **Guide RNA design:** CRISPR gRNAs and other synthetic RNAs require specific structural features
* **RNA therapeutics:** siRNA, ASO, and mRNA vaccine design all consider secondary structure
* **Viral RNA:** Understanding viral RNA structures aids in understanding replication and drug targeting

**Scientific foundation:**
ViennaRNA uses a dynamic programming algorithm based on the [nearest-neighbor](https://en.wikipedia.org/wiki/Nucleic_acid_thermodynamics#Nearest-neighbor_method) thermodynamic model. Energy parameters (Turner 2004 for RNA, Mathews 2004 for DNA) describe the free energy contributions of:

* **Base pair stacking:** The stabilizing interactions between adjacent base pairs
* **Loop energies:** Penalties for hairpin loops, internal loops, bulges, and multiloops
* **Dangling ends:** Contributions from unpaired nucleotides adjacent to helices

The algorithm finds the structure with the lowest total free energy (most stable).

## Available tools

In [2]:
display_available_tools("viennarna")

- **`run_viennarna()`** — RNA secondary structure prediction using ViennaRNA MFE folding

### `run_viennarna`

ViennaRNA computes the minimum free energy secondary structure of an RNA sequence using the thermodynamic nearest-neighbor model. The output structure uses dot-bracket notation where '.' denotes an unpaired base and matched '(' ')' pairs denote base pairs. More negative MFE values indicate more thermodynamically stable folds. The five sequences below span a range of structural behaviors: GC-rich stems with UUUU loops (highly stable), mixed stems, and unstructured AU-rich sequences (near-zero MFE), making them a useful comparison set for understanding how sequence composition drives fold stability.

In [3]:
from proto_tools import ViennaRNAInput, ViennaRNAConfig, run_viennarna

In [4]:
# Display input API reference
display_api_reference("viennarna", "input", "run_viennarna")

# Five RNA sequences spanning a range of structural behaviors
inputs = ViennaRNAInput(sequences=[
    "GCGCUUUUGCGC",    # GC-rich stem with UUUU loop (stable hairpin)
    "GGGGAAACCCC",     # G-rich stem with AAA loop
    "CGCGAAACGCG",     # Alternating CG stem with AAA loop
    "UUUUAAAAAUUUUU",  # Poly-U/A stretch (expected to be unstructured)
    "GCUAGCUAGC",      # Mixed sequence with moderate folding potential
])

**Input** — `ViennaRNAInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | List of RNA sequences to fold. Each sequence should contain only valid RNA nucleotides (A, U, G, C) or DNA nucleotides (A, T, G, C) which will be automatically converted to RNA (T -> U). Lowercase letters are also accepted. |

In [5]:
# Display config API reference
display_api_reference("viennarna", "config", "run_viennarna")

# Configure ViennaRNA folding at physiological temperature
config = ViennaRNAConfig(
    temperature=37.0,  # Physiological temperature in Celsius
)

**Config** — `ViennaRNAConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `temperature` | `number` | `37.0` | Temperature in Celsius for energy calculations. Affects the thermodynamic parameters used in folding. Default: 37.0 (physiological temperature). |
| `use_dna_params` | `boolean` | `False` | Whether to use DNA energy parameters instead of RNA parameters. When True, loads DNA\_Mathews2004 parameters. |
| `no_lonely_pairs` | `boolean` | `False` | Disallow lonely base pairs (helices of length 1). This can reduce artifacts in structure prediction. Default: False. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the prediction
result = run_viennarna(inputs, config)

Running run_viennarna [00:00]

In [7]:
# Display output API reference
display_api_reference("viennarna", "output", "run_viennarna")

# Display predicted structures and MFE for each sequence
for i, res in enumerate(result.results):
    print(f"Sequence {i + 1}: {res.sequence}")
    print(f"  Structure: {res.structure}")
    print(f"  MFE:       {res.mfe:.2f} kcal/mol")
    print()

**Output** — `ViennaRNAOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `List[ViennaRNAResult]` | required | List of fold results, one per input sequence. Each result contains the sequence, predicted structure in dot-bracket notation, and the minimum free energy. |
| `sequence` | `string` | required | The input RNA sequence. |
| `structure` | `string` |  | Predicted secondary structure in dot-bracket notation. |
| `mfe` | `number` |  | Minimum free energy in kcal/mol. |

Sequence 1: GCGCUUUUGCGC
  Structure: ((((....))))
  MFE:       -5.70 kcal/mol

Sequence 2: GGGGAAACCCC
  Structure: ((((...))))
  MFE:       -4.50 kcal/mol

Sequence 3: CGCGAAACGCG
  Structure: ((((...))))
  MFE:       -2.80 kcal/mol

Sequence 4: UUUUAAAAAUUUUU
  Structure: ..............
  MFE:       0.00 kcal/mol

Sequence 5: GCUAGCUAGC
  Structure: (((....)))
  MFE:       -0.20 kcal/mol



## Export Results

Folding results can be exported to CSV, JSON, or FASTA format for downstream analysis. The CSV format produces a simple table with sequence, structure, and MFE columns that can be loaded into a spreadsheet or any data-processing pipeline. The FASTA format includes the MFE as a sequence header annotation alongside the dot-bracket structure.

In [8]:
from pathlib import Path

# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to CSV
result.export("viennarna_results", export_path=output_dir, file_format="csv")